# Fase 4 — Análisis Exploratorio de Datos
## TechOps MRO Analytics · Business Analytics Engineer Assessment

El objetivo es reunir evidencia suficiente para responder las cinco preguntas del caso.
La limpieza y el marco de métricas de las fases anteriores ya están incorporados.

| Sección | Propósito |
|---|---|
| Estadística descriptiva | Contexto numérico de partida |
| Distribuciones | Forma y concentración de las variables clave |
| Outliers | Variabilidad y valores extremos por segmento |
| Correlaciones | Relaciones entre variables operativas a nivel WO |
| Evolución temporal | Tendencia de indicadores en el período analizado |
| Hallazgos | Síntesis para la presentación ejecutiva |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

Path("graficas").mkdir(exist_ok=True)

CORAL = "#D85A30"
TEAL  = "#1D9E75"
AMBAR = "#BA7517"
PURP  = "#7F77DD"
GRIS  = "#888780"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.25,
    "grid.linestyle":   "--",
    "font.size":        11,
})

In [ ]:
BASE = Path("data/cleaned")

av  = pd.read_parquet(BASE / "aircraft_visits.parquet")
wo  = pd.read_parquet(BASE / "work_orders_productividad.parquet")
lt  = pd.read_parquet(BASE / "labor_transactions.parquet")
de  = pd.read_parquet(BASE / "delay_events.parquet")
qf  = pd.read_parquet(BASE / "quality_findings.parquet")

wo["temporal_anchor"] = pd.to_datetime(wo["temporal_anchor"])

print(f"Cargados: {len(av):,} visitas | {len(wo):,} órdenes | "
      f"{len(lt):,} transacciones | {len(de):,} retrasos | {len(qf):,} hallazgos")

In [ ]:
# columnas derivadas en LT que usaremos en varias secciones
lt["costo_ot"]       = lt["Overtime_Hours"] * lt["Overtime_Rate"]
lt["horas_tot"]      = lt["Regular_Hours"]  + lt["Overtime_Hours"]
lt["ratio_ot_txn"]   = lt["Overtime_Hours"] / lt["horas_tot"].replace(0, np.nan)

# LT al nivel de orden de trabajo
lt_wo = (lt.groupby("Work_Order_ID")
    .agg(costo_total=("Labor_Cost","sum"),
         horas_ot=("Overtime_Hours","sum"),
         horas_tot=("horas_tot","sum"))
    .reset_index()
)
lt_wo["ratio_ot"] = lt_wo["horas_ot"] / lt_wo["horas_tot"].replace(0, np.nan)

# QF al nivel de orden de trabajo
qf_wo = (qf.groupby("Work_Order_ID")
    .agg(horas_rework=("Rework_Hours","sum"), n_findings=("Finding_ID","count"))
    .reset_index()
)

# tabla maestra WO — agregar antes de unir para evitar fan-out
wo_m = (wo
    .merge(lt_wo, on="Work_Order_ID", how="left")
    .merge(qf_wo, on="Work_Order_ID", how="left")
)
wo_m[["horas_rework","n_findings"]] = wo_m[["horas_rework","n_findings"]].fillna(0)
wo_m["efficiency_ratio"] = wo_m["Planned_Hours"] / wo_m["Actual_Hours"].replace(0, np.nan)
wo_m["es_overrun"]       = (wo_m["Actual_Hours"] > wo_m["Planned_Hours"]).astype(int)
wo_m["periodo"]          = wo_m["temporal_anchor"].dt.to_period("M")

# tabla de visitas
de_av = de.groupby("Visit_ID")["Delay_Hours"].sum().reset_index(name="horas_retraso")
wo_av = (wo_m.groupby("Visit_ID")
    .agg(costo_v=("costo_total","sum"), horas_ot_v=("horas_ot","sum"),
         horas_tot_v=("horas_tot","sum"), rework_v=("horas_rework","sum"))
    .reset_index()
    .merge(av[["Visit_ID","TAT_days","TAT_variance_days","Check_Type","Customer","Station"]],
           on="Visit_ID", how="left")
    .merge(de_av, on="Visit_ID", how="left")
)
wo_av["horas_retraso"] = wo_av["horas_retraso"].fillna(0)

print(f"Tabla WO: {len(wo_m):,} filas | Tabla visita: {len(wo_av):,} filas")

## 1. Estadística descriptiva

In [ ]:
costo_tot    = lt["Labor_Cost"].sum()
costo_ot     = lt["costo_ot"].sum()
ratio_ot_hrs = lt["Overtime_Hours"].sum() / lt["horas_tot"].sum()
er_global    = wo_m["Planned_Hours"].sum() / wo_m["Actual_Hours"].sum()
overrun_rate = wo_m["es_overrun"].mean()
rework_rate  = qf["Rework_Hours"].sum() / wo_m["Actual_Hours"].sum()
pct_tarde    = (av["TAT_variance_days"] > 0).mean()

indicadores = {
    "Costo laboral total ($)"           : f"${costo_tot:>14,.0f}",
    "Costo overtime ($)"                : f"${costo_ot:>14,.0f}  ({costo_ot/costo_tot*100:.1f}%)",
    "Ratio OT sobre horas totales"      : f"{ratio_ot_hrs*100:>15.1f}%",
    "Efficiency ratio global"           : f"{er_global:>18.3f}",
    "Órdenes con sobrerun"              : f"{overrun_rate*100:>15.1f}%",
    "Rework rate (h rework / h real)"   : f"{rework_rate*100:>15.2f}%",
    "Visitas entregadas fuera de plazo" : f"{pct_tarde*100:>15.1f}%",
}

print("Indicadores del período analizado")
print()
for k, v in indicadores.items():
    print(f"  {k:<40}: {v}")

## 2. Distribuciones

Efficiency ratio (por complejidad), ratio OT por orden y desvío de TAT son las tres variables
con mayor poder explicativo para las preguntas del caso.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

colores_comp = {"Low": TEAL, "Medium": AMBAR, "High": CORAL}
for comp, grp in wo_m.groupby("Complexity"):
    vals = grp["efficiency_ratio"].dropna().clip(0.2, 2.5)
    axes[0].hist(vals, bins=40, alpha=0.5, density=True,
                 color=colores_comp.get(comp, GRIS), label=comp)
axes[0].axvline(1, lw=1, ls="--", color="black", alpha=0.6)
axes[0].set_title("Efficiency ratio por complejidad")
axes[0].set_xlabel("Planned / Actual Hours")
axes[0].set_ylabel("Densidad")
axes[0].legend(frameon=False, fontsize=9)

ot_vals = wo_m["ratio_ot"].dropna().clip(0, 1)
axes[1].hist(ot_vals, bins=40, color=CORAL, alpha=0.8, density=True)
axes[1].axvline(ot_vals.mean(), lw=1.5, ls="--", color="black",
                label=f"Media: {ot_vals.mean():.2f}")
axes[1].set_title("Ratio OT por orden de trabajo")
axes[1].set_xlabel("OT Hours / Total Hours")
axes[1].set_ylabel("Densidad")
axes[1].legend(frameon=False, fontsize=9)

tat = av["TAT_variance_days"]
axes[2].hist(tat, bins=35, color=AMBAR, alpha=0.8, density=True)
axes[2].axvline(0,          lw=1,   ls="--", color="black", alpha=0.5)
axes[2].axvline(tat.mean(), lw=1.5, ls="--", color=CORAL,
                label=f"Media: {tat.mean():.1f} d")
axes[2].set_title("Desvío de TAT vs. plan")
axes[2].set_xlabel("Días (positivo = tardío)")
axes[2].set_ylabel("Densidad")
axes[2].legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig("graficas/histogramas.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Outliers

El efficiency ratio segmentado por complejidad y el ratio OT por turno revelan
si las distribuciones operativas son homogéneas o si hay segmentos que concentran el problema.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_bp = [TEAL, AMBAR, CORAL]

orden_comp = ["Low", "Medium", "High"]
datos_comp = [wo_m[wo_m["Complexity"]==c]["efficiency_ratio"].dropna().clip(0.1, 3.0)
              for c in orden_comp]
bp1 = axes[0].boxplot(datos_comp, labels=orden_comp, patch_artist=True, notch=False,
                       medianprops={"color":"white","linewidth":2},
                       whiskerprops={"linewidth":1}, capprops={"linewidth":1},
                       flierprops={"marker":".","markersize":3,"alpha":0.3})
for p, c in zip(bp1["boxes"], colores_bp):
    p.set_facecolor(c); p.set_alpha(0.7)
axes[0].axhline(1, lw=1, ls="--", color="black", alpha=0.4)
axes[0].set_title("Efficiency ratio por nivel de complejidad")
axes[0].set_xlabel("Complejidad")
axes[0].set_ylabel("Planned / Actual Hours  (>1 = dentro del plan)")

orden_turno  = ["Day",    "Swing",  "Night"]
etiq_turno   = ["Diurno", "Swing",  "Nocturno"]
datos_turno  = [lt[lt["Shift"]==t]["ratio_ot_txn"].dropna() for t in orden_turno]
bp2 = axes[1].boxplot(datos_turno, labels=etiq_turno, patch_artist=True, notch=False,
                       medianprops={"color":"white","linewidth":2},
                       whiskerprops={"linewidth":1}, capprops={"linewidth":1},
                       flierprops={"marker":".","markersize":3,"alpha":0.3})
for p, c in zip(bp2["boxes"], colores_bp):
    p.set_facecolor(c); p.set_alpha(0.7)
axes[1].set_title("Ratio OT por turno")
axes[1].set_xlabel("Turno")
axes[1].set_ylabel("OT Hours / Total Hours")

plt.tight_layout()
plt.savefig("graficas/boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Correlaciones

Matriz a nivel orden de trabajo. La relación entre sobrerun y ratio OT es el mecanismo
central que conecta la ejecución operativa con el deterioro de costos.

In [ ]:
vars_corr = (wo_m
    .dropna(subset=["efficiency_ratio","ratio_ot"])
    [["Planned_Hours","Actual_Hours","es_overrun","ratio_ot","horas_rework","n_findings"]]
    .rename(columns={
        "Planned_Hours": "h_plan",
        "Actual_Hours":  "h_real",
        "es_overrun":    "sobrerun",
        "ratio_ot":      "ratio_ot",
        "horas_rework":  "h_rework",
        "n_findings":    "hallazgos",
    })
    .astype(float)
)

corr = vars_corr.corr()
n    = len(corr)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(n)); ax.set_xticklabels(corr.columns, rotation=40, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(corr.columns)
for i in range(n):
    for j in range(n):
        val = corr.iloc[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=9, color="white" if abs(val) > 0.55 else "black")
ax.set_title("Matriz de correlación — nivel orden de trabajo")
plt.tight_layout()
plt.savefig("graficas/correlacion.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Evolución temporal y desglose estructural

Agregación mensual por `temporal_anchor` (ver supuesto A04).
Los meses de mayo–junio reflejan períodos parciales: visitas tardías con pocas órdenes cerradas.

In [ ]:
meses_es = {"01":"Ene","02":"Feb","03":"Mar","04":"Abr",
            "05":"May","06":"Jun","07":"Jul","08":"Ago",
            "09":"Sep","10":"Oct","11":"Nov","12":"Dic"}

def label_mes(p):
    y, m = str(p).split("-")
    return f"{meses_es.get(m, m)} {y}"

mensual_lt = (lt
    .merge(wo_m[["Work_Order_ID","periodo"]], on="Work_Order_ID", how="left")
    .dropna(subset=["periodo"])
    .groupby("periodo")
    .agg(costo=("Labor_Cost","sum"), costo_ot=("costo_ot","sum"))
    .reset_index()
    .sort_values("periodo")
)
mensual_lt["pct_ot"]  = mensual_lt["costo_ot"] / mensual_lt["costo"]
mensual_lt["mes_str"] = mensual_lt["periodo"].map(label_mes)

mensual_wo = (wo_m.groupby("periodo")
    .agg(er_medio=("efficiency_ratio","mean"), overrun_rate=("es_overrun","mean"),
         hrw=("horas_rework","sum"), hreal=("Actual_Hours","sum"))
    .reset_index()
    .sort_values("periodo")
)
mensual_wo["rework_rate"] = mensual_wo["hrw"] / mensual_wo["hreal"]
mensual_wo["mes_str"]     = mensual_wo["periodo"].map(label_mes)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
x  = np.arange(len(mensual_lt))
x2 = np.arange(len(mensual_wo))

# Costo mensual apilado + línea % OT
reg = (mensual_lt["costo"] - mensual_lt["costo_ot"]) / 1e6
ot  =  mensual_lt["costo_ot"] / 1e6
axes[0].bar(x, reg, color=TEAL,  alpha=0.8, label="Regular")
axes[0].bar(x, ot,  color=CORAL, alpha=0.8, bottom=reg, label="Overtime")
ax0b = axes[0].twinx()
ax0b.plot(x, mensual_lt["pct_ot"]*100, color="black", lw=1.8, marker="o", ms=4, label="% OT")
ax0b.set_ylabel("% costo OT", fontsize=10); ax0b.set_ylim(0, 60)
ax0b.spines["top"].set_visible(False)
axes[0].set_xticks(x); axes[0].set_xticklabels(mensual_lt["mes_str"], rotation=30)
axes[0].set_ylabel("Costo laboral (M$)")
axes[0].set_title("Costo laboral mensual — regular vs. overtime")
h0,  l0  = axes[0].get_legend_handles_labels()
h0b, l0b = ax0b.get_legend_handles_labels()
axes[0].legend(h0+h0b, l0+l0b, frameon=False, fontsize=9, loc="upper left")

# Efficiency ratio mensual + % sobrerun
axes[1].plot(x2, mensual_wo["er_medio"], color=TEAL, lw=2, marker="o", ms=5,
             label="Efficiency ratio")
axes[1].axhline(1, lw=1, ls="--", color="black", alpha=0.4)
ax1b = axes[1].twinx()
ax1b.bar(x2, mensual_wo["overrun_rate"]*100, color=CORAL, alpha=0.25, label="% sobrerun")
ax1b.set_ylabel("% órdenes con sobrerun", fontsize=10)
ax1b.spines["top"].set_visible(False)
axes[1].set_xticks(x2); axes[1].set_xticklabels(mensual_wo["mes_str"], rotation=30)
axes[1].set_ylabel("Efficiency ratio promedio")
axes[1].set_title("Efficiency ratio y tasa de sobrerun mensual")
h1,  l1  = axes[1].get_legend_handles_labels()
h1b, l1b = ax1b.get_legend_handles_labels()
axes[1].legend(h1+h1b, l1+l1b, frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig("graficas/temporal.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Horas de retraso por área responsable
delay_area = de.groupby("Responsible_Area")["Delay_Hours"].sum().sort_values()
axes[0].barh(delay_area.index, delay_area.values / 1000, color=AMBAR, alpha=0.8)
axes[0].set_xlabel("Horas de retraso (miles)")
axes[0].set_title("Retrasos por área responsable")
for i, v in enumerate(delay_area.values / 1000):
    axes[0].text(v + 0.1, i, f"{v:.1f}k", va="center", fontsize=9, color=GRIS)

# Ratio OT por especialidad
ot_skill = (lt.groupby("Standard_Skill")[["Overtime_Hours","horas_tot"]].sum()
              .assign(ratio=lambda d: d["Overtime_Hours"]/d["horas_tot"]*100)
              .sort_values("ratio"))
axes[1].barh(ot_skill.index, ot_skill["ratio"], color=CORAL, alpha=0.8)
axes[1].axvline(ot_skill["ratio"].mean(), lw=1.5, ls="--", color="black", alpha=0.6,
                label=f"Media red: {ot_skill['ratio'].mean():.0f}%")
axes[1].set_xlabel("Ratio OT (%)")
axes[1].set_title("Ratio de overtime por especialidad")
axes[1].legend(frameon=False, fontsize=9)
for i, v in enumerate(ot_skill["ratio"]):
    axes[1].text(v + 0.2, i, f"{v:.0f}%", va="center", fontsize=9, color=GRIS)

plt.tight_layout()
plt.savefig("graficas/desglose_estructural.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Hallazgos principales

Cada hallazgo está computado desde los datos y redactado para ser utilizable
directamente en la presentación ejecutiva.

In [ ]:
costo_ot_pct = lt["costo_ot"].sum() / lt["Labor_Cost"].sum() * 100

turno_stats  = (lt.groupby("Shift")[["Overtime_Hours","horas_tot"]].sum()
                  .assign(ratio_ot=lambda d: d["Overtime_Hours"]/d["horas_tot"]*100))
ot_noche   = turno_stats.loc["Night", "ratio_ot"] if "Night"  in turno_stats.index else 0
ot_diurno  = turno_stats.loc["Day",   "ratio_ot"] if "Day"    in turno_stats.index else 0

overrun_global   = wo_m["es_overrun"].mean() * 100
overrun_por_comp = (wo_m.groupby("Complexity")["es_overrun"].mean() * 100).to_dict()

er_mediana_por_comp = wo_m.groupby("Complexity")["efficiency_ratio"].median().to_dict()

rework_hrs        = qf["Rework_Hours"].sum()
rework_pct        = rework_hrs / wo_m["Actual_Hours"].sum() * 100
costo_rework_est  = rework_hrs * lt["Regular_Rate"].mean()

pct_tarde         = (av["TAT_variance_days"] > 0).mean() * 100
tat_medio_tarde   = av[av["TAT_variance_days"] > 0]["TAT_variance_days"].mean()

ot_por_skill = (lt.groupby("Standard_Skill")[["Overtime_Hours","horas_tot"]].sum()
                  .assign(ratio=lambda d: d["Overtime_Hours"]/d["horas_tot"]*100)
                  .sort_values("ratio", ascending=False))
skill_top       = ot_por_skill.index[0]
skill_top_ratio = ot_por_skill["ratio"].iloc[0]

delay_area = de.groupby("Responsible_Area")["Delay_Hours"].sum().sort_values(ascending=False)
top2_pct   = delay_area.head(2).sum() / delay_area.sum() * 100
top2_areas = delay_area.head(2).index.tolist()

# tendencia sobre meses con datos completos (excluir may-jun parciales)
meses_completos = mensual_wo[mensual_wo["hreal"] > mensual_wo["hreal"].quantile(0.3)]
er_inicio = meses_completos["er_medio"].iloc[0]
er_fin    = meses_completos["er_medio"].iloc[-1]
delta_er  = er_fin - er_inicio
tendencia = "leve mejora" if delta_er > 0.02 else "sin variación significativa"

print("Hallazgos principales")
print("─" * 60)
print()
print(f"1. El {costo_ot_pct:.0f}% del costo laboral total corresponde a overtime; el turno"
      f" nocturno registra {ot_noche:.0f}% de ratio OT frente al {ot_diurno:.0f}% del diurno,"
      f" lo que señala un problema estructural de dotación en ese turno, no picos ocasionales.")
print()
print(f"2. El {overrun_global:.0f}% de las órdenes supera las horas planeadas de forma"
      f" uniforme entre niveles de complejidad (High: {overrun_por_comp.get('High',0):.0f}%,"
      f" Low: {overrun_por_comp.get('Low',0):.0f}%), lo que descarta el efecto de mix y apunta"
      f" a un problema sistémico en los estándares de estimación.")
print()
print(f"3. El efficiency ratio global es"
      f" {wo_m['Planned_Hours'].sum()/wo_m['Actual_Hours'].sum():.2f}, prácticamente en el"
      f" umbral del plan (1.0). La supuesta mejora de productividad no es visible en los datos.")
print()
print(f"4. El rework acumula {rework_hrs:,.0f} horas ({rework_pct:.1f}% del total trabajado),"
      f" equivalentes a un costo estimado de ${costo_rework_est:,.0f} al aplicar la tarifa promedio.")
print()
print(f"5. El {pct_tarde:.0f}% de las visitas se entrega fuera del plazo comprometido,"
      f" con un desvío promedio de {tat_medio_tarde:.1f} días en los casos tardíos.")
print()
print(f"6. La especialidad '{skill_top}' registra el mayor ratio de overtime ({skill_top_ratio:.0f}%),"
      f" indicando un cuello de botella de disponibilidad en esa habilidad específica.")
print()
print(f"7. '{top2_areas[0]}' y '{top2_areas[1]}' concentran el {top2_pct:.0f}% de las horas"
      f" de retraso, siendo los principales inhibidores del flujo operativo.")
print()
print(f"8. La evolución mensual del efficiency ratio muestra {tendencia} entre enero y abril;"
      f" el costo total creció de forma sostenida en ese mismo período, lo que confirma"
      f" la paradoja central del caso.")